# Bivariate Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/02-exploratory-data-analysis/03_bivariate_analysis.ipynb)

## Learning Objectives
- Understand relationships between two variables
- Master visualization techniques for different variable type combinations
- Learn to identify correlation vs causation
- Discover patterns and dependencies between features

---

## 1. What is Bivariate Analysis?

**Bivariate Analysis** = Analyzing **TWO variables together** to find relationships

**Goals:**
- 🔗 Discover associations between variables
- 📊 Understand how one variable affects another
- 🎯 Identify which features relate to the target
- 🧠 Generate hypotheses for modeling

**Three main combinations:**
1. **Numerical vs Numerical**: Scatter plots, correlation
2. **Categorical vs Numerical**: Box plots, grouped statistics
3. **Categorical vs Categorical**: Cross-tabulation, stacked bars

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 2. Dataset: E-commerce Purchase Behavior

In [ ]:
# Create synthetic e-commerce dataset
np.random.seed(42)
n = 800

# Generate correlated features
time_on_site = np.random.gamma(5, 2, n)  # Minutes
pages_viewed = np.random.poisson(time_on_site * 1.5, n)
previous_purchases = np.random.poisson(3, n)

# Purchase amount depends on multiple factors
purchase_amount = (
    time_on_site * 15 +
    pages_viewed * 8 +
    previous_purchases * 25 +
    np.random.normal(0, 30, n)
).clip(0)

data = {
    'customer_id': range(1, n+1),
    'age': np.random.normal(35, 12, n).clip(18, 70).astype(int),
    'time_on_site_min': time_on_site.round(1),
    'pages_viewed': pages_viewed,
    'previous_purchases': previous_purchases,
    'purchase_amount': purchase_amount.round(2),
    'device_type': np.random.choice(['Mobile', 'Desktop', 'Tablet'], n, p=[0.5, 0.35, 0.15]),
    'membership': np.random.choice(['Free', 'Premium'], n, p=[0.7, 0.3]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'made_purchase': np.random.choice(['Yes', 'No'], n, p=[0.4, 0.6])
}

# Premium members spend more
premium_mask = data['membership'] == 'Premium'
data['purchase_amount'][premium_mask] *= 1.3

df = pd.DataFrame(data)

print("✅ E-commerce dataset created!")
print(f"Shape: {df.shape}")
df.head(10)

## 3. Numerical vs Numerical

### 3.1 Scatter Plot - The Foundation

In [ ]:
# Basic scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df['time_on_site_min'], df['purchase_amount'], alpha=0.5, edgecolors='black', s=50)
plt.xlabel('Time on Site (minutes)', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Time on Site vs Purchase Amount', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
corr = df['time_on_site_min'].corr(df['purchase_amount'])
print(f"\n🔗 Correlation: {corr:.3f}")

if abs(corr) > 0.7:
    print("✅ Strong correlation")
elif abs(corr) > 0.4:
    print("⚡ Moderate correlation")
else:
    print("⚠️ Weak correlation")

### 3.2 Enhanced Scatter Plots

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='time_on_site_min', y='purchase_amount',
            scatter_kws={'alpha': 0.5, 's': 50},
            line_kws={'color': 'red', 'linewidth': 2})
plt.xlabel('Time on Site (minutes)', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Time on Site vs Purchase Amount (with Trend Line)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Hexbin plot (for large datasets - shows density)
plt.figure(figsize=(10, 6))
plt.hexbin(df['pages_viewed'], df['purchase_amount'], gridsize=20, cmap='YlOrRd', mincnt=1)
plt.colorbar(label='Count')
plt.xlabel('Pages Viewed', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Pages Viewed vs Purchase Amount (Density)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Pair Plot - Multiple Numerical Variables

In [ ]:
# Select key numerical columns
num_cols = ['time_on_site_min', 'pages_viewed', 'previous_purchases', 'purchase_amount']

# Create pair plot
sns.pairplot(df[num_cols], diag_kind='kde', plot_kws={'alpha': 0.5, 's': 30},
             diag_kws={'linewidth': 2})
plt.suptitle('Pair Plot: All Numerical Variables', y=1.01, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Diagonal = Distribution of each variable")
print("   Off-diagonal = Relationships between pairs")

### 3.4 Correlation Matrix

In [ ]:
# Calculate correlation matrix
corr_matrix = df[num_cols].corr()

print("🔗 Correlation Matrix:")
print(corr_matrix.round(3))

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Red = Positive correlation (both increase together)")
print("   Blue = Negative correlation (one increases, other decreases)")
print("   White = No correlation")

## 4. Categorical vs Numerical

### 4.1 Box Plot - Distribution Comparison

In [ ]:
# Box plot: Device type vs Purchase amount
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='device_type', y='purchase_amount', palette='Set2')
plt.xlabel('Device Type', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Purchase Amount by Device Type', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Summary statistics by group
print("\n📊 Purchase Amount by Device Type:")
print(df.groupby('device_type')['purchase_amount'].describe().round(2))

### 4.2 Violin Plot - Richer Distribution View

In [ ]:
# Violin plot: Membership vs Purchase amount
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='membership', y='purchase_amount', palette='muted', inner='quartile')
plt.xlabel('Membership Type', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Purchase Amount by Membership Type', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n📊 Average Purchase by Membership:")
membership_stats = df.groupby('membership')['purchase_amount'].agg(['mean', 'median', 'std', 'count'])
print(membership_stats.round(2))

### 4.3 Bar Plot - Mean Comparison

In [ ]:
# Bar plot with error bars
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='region', y='purchase_amount', palette='viridis',
            errorbar='sd', capsize=0.1)
plt.xlabel('Region', fontsize=12)
plt.ylabel('Average Purchase Amount ($)', fontsize=12)
plt.title('Average Purchase Amount by Region (with Std Dev)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n📊 Purchase Amount by Region:")
print(df.groupby('region')['purchase_amount'].agg(['mean', 'std', 'count']).round(2))

### 4.4 Statistical Test - Is the difference significant?

In [ ]:
# T-test: Compare Premium vs Free membership
premium = df[df['membership'] == 'Premium']['purchase_amount']
free = df[df['membership'] == 'Free']['purchase_amount']

t_stat, p_value = stats.ttest_ind(premium, free)

print("📊 T-Test: Premium vs Free Membership")
print(f"Premium Mean: ${premium.mean():.2f}")
print(f"Free Mean: ${free.mean():.2f}")
print(f"Difference: ${premium.mean() - free.mean():.2f}")
print(f"\nT-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Statistically significant difference! (p < 0.05)")
    print("   → Premium members spend significantly more")
else:
    print("\n❌ No significant difference (p >= 0.05)")

## 5. Categorical vs Categorical

### 5.1 Cross-Tabulation (Contingency Table)

In [ ]:
# Cross-tabulation: Device Type vs Made Purchase
crosstab = pd.crosstab(df['device_type'], df['made_purchase'], margins=True)

print("📊 Cross-Tabulation: Device Type vs Made Purchase")
print(crosstab)

# Normalized (percentages)
crosstab_pct = pd.crosstab(df['device_type'], df['made_purchase'], normalize='index') * 100

print("\n📊 Percentages (by Device Type):")
print(crosstab_pct.round(2))

### 5.2 Stacked Bar Chart

In [ ]:
# Stacked bar chart
crosstab_counts = pd.crosstab(df['device_type'], df['made_purchase'])

crosstab_counts.plot(kind='bar', stacked=True, figsize=(10, 6),
                      color=['#ff9999', '#66b3ff'], edgecolor='black')
plt.xlabel('Device Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Device Type vs Purchase Decision (Stacked)', fontsize=14, fontweight='bold')
plt.legend(title='Made Purchase', loc='upper right')
plt.xticks(rotation=0)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart (side-by-side)
crosstab_counts.plot(kind='bar', stacked=False, figsize=(10, 6),
                      color=['#ff9999', '#66b3ff'], edgecolor='black')
plt.xlabel('Device Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Device Type vs Purchase Decision (Grouped)', fontsize=14, fontweight='bold')
plt.legend(title='Made Purchase', loc='upper right')
plt.xticks(rotation=0)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 5.3 Heatmap Visualization

In [ ]:
# Heatmap of cross-tabulation
plt.figure(figsize=(8, 5))
sns.heatmap(crosstab_pct, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'Percentage'})
plt.xlabel('Made Purchase', fontsize=12)
plt.ylabel('Device Type', fontsize=12)
plt.title('Purchase Rate by Device Type (%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 Chi-Square Test - Are variables independent?

In [ ]:
# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(crosstab_counts)

print("📊 Chi-Square Test: Device Type vs Made Purchase")
print(f"Chi-square statistic: {chi2:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\n✅ Variables are dependent! (p < 0.05)")
    print("   → Device type influences purchase decision")
else:
    print("\n❌ Variables are independent (p >= 0.05)")
    print("   → Device type doesn't affect purchase decision")

## 6. Correlation vs Causation ⚠️

**CRITICAL CONCEPT**: Correlation ≠ Causation!

### Examples:

1. **Correlation found**: Ice cream sales ↔ Drowning deaths
   - **Real cause**: Both increase in summer (confounding variable)

2. **Correlation found**: Time on site ↔ Purchase amount
   - **Possible interpretations**:
     - ✅ More time → More likely to buy (causation)
     - ⚠️ Interested buyers → Spend more time (reverse causation)
     - ⚠️ Product quality → Both (common cause)

### How to establish causation:
- 🧪 Randomized controlled experiments
- 📊 Longitudinal studies
- 🧠 Domain knowledge and theory
- 🔍 Eliminate confounding variables

In [ ]:
# Example: Spurious correlation
# Let's create two unrelated variables that happen to correlate

np.random.seed(123)
trend = np.linspace(0, 10, 100)
noise1 = np.random.normal(0, 1, 100)
noise2 = np.random.normal(0, 1, 100)

var1 = trend + noise1  # Both follow the same trend
var2 = trend + noise2

spurious_corr = np.corrcoef(var1, var2)[0, 1]

plt.figure(figsize=(10, 6))
plt.scatter(var1, var2, alpha=0.6, s=50)
plt.xlabel('Variable 1 (e.g., Number of Nicolas Cage movies)', fontsize=11)
plt.ylabel('Variable 2 (e.g., Pool drownings)', fontsize=11)
plt.title(f'Spurious Correlation Example\nCorrelation: {spurious_corr:.3f}', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"⚠️ Strong correlation ({spurious_corr:.3f}) but NO causation!")
print("   Both variables just happened to increase over time.")

## 7. Bivariate Analysis Decision Tree

```
What types of variables?
    |
    ├─ NUMERICAL vs NUMERICAL
    │   ├─ Scatter plot (with regression line)
    │   ├─ Calculate correlation (Pearson/Spearman)
    │   ├─ Pair plot (for multiple variables)
    │   └─ Hexbin plot (for large datasets)
    │
    ├─ CATEGORICAL vs NUMERICAL
    │   ├─ Box plot / Violin plot
    │   ├─ Bar plot (means with error bars)
    │   ├─ Group statistics (mean, median, std)
    │   └─ T-test / ANOVA (if testing difference)
    │
    └─ CATEGORICAL vs CATEGORICAL
        ├─ Cross-tabulation (frequencies)
        ├─ Stacked/Grouped bar chart
        ├─ Heatmap of percentages
        └─ Chi-square test (independence)
```

## 8. Advanced: Multiple Relationships

In [ ]:
# Scatter plot colored by category
plt.figure(figsize=(12, 6))

for device in df['device_type'].unique():
    mask = df['device_type'] == device
    plt.scatter(df[mask]['time_on_site_min'], df[mask]['purchase_amount'],
                label=device, alpha=0.6, s=50)

plt.xlabel('Time on Site (minutes)', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.title('Time vs Purchase Amount by Device Type', fontsize=14, fontweight='bold')
plt.legend(title='Device Type', loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🔗 Correlation by Device Type:")
for device in df['device_type'].unique():
    mask = df['device_type'] == device
    corr = df[mask]['time_on_site_min'].corr(df[mask]['purchase_amount'])
    print(f"{device}: {corr:.3f}")

## 9. Your Turn! 💪

**Exercise**: Explore these relationships in the dataset:
1. **Numerical vs Numerical**: Age vs Purchase Amount
2. **Categorical vs Numerical**: Region vs Time on Site
3. **Categorical vs Categorical**: Membership vs Made Purchase

For each:
- Create appropriate visualizations
- Calculate relevant statistics
- Write 2-3 insights

In [ ]:
# Your code here

---

## Key Takeaways 🎯

1. **Numerical vs Numerical**: Scatter plots + correlation (watch for non-linear relationships)
2. **Categorical vs Numerical**: Box plots + group statistics + statistical tests
3. **Categorical vs Categorical**: Cross-tabs + stacked bars + chi-square test
4. **Correlation ≠ Causation**: Always consider confounding variables
5. **Visualize relationships**: Charts reveal patterns statistics might miss

**Next**: Deep dive into correlation analysis! 📈